# Lab 21-08 · Transceiver Architectures

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import trx


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · A frequency plan

*Predict* the image and half-IF spurs of a 2.45 GHz receiver with a 2.75 GHz LO, and the preselector order for 60 dB.

In [2]:
f_img, f_lo = trx.image(2.45e9, 300e6)
print(f"LO {f_lo/1e9:.2f} GHz, image {f_img/1e9:.2f} GHz, order for 60 dB: {trx.order_for_rejection(f_img, 2.45e9, 100e6, 60)}")
for m, n, f in trx.spurs(f_lo, 300e6, 1e6, 1.0e9, 4.0e9, 4, 4):
    print(f"  (m, n) = ({m}, {n}): {f/1e9:.4f} GHz, preselector {trx.butterworth_bp_atten_db(f, 2.45e9, 100e6, 3):6.1f} dB")
s = ngspice("preselect.cir")
print({k: round(s[k], 2) for k in ("aim", "asp1", "asp2")})
f = np.linspace(2.0e9, 3.3e9, 1000)
plt.figure(); plt.plot(f / 1e9, [-trx.butterworth_bp_atten_db(x, 2.45e9, 100e6, 3) for x in f])
for x, lab_ in ((2.45e9, "wanted"), (2.75e9, "LO"), (3.05e9, "image"), (2.60e9, "half-IF"), (2.90e9, "half-IF")):
    plt.axvline(x / 1e9, ls=":", c="k"); plt.text(x / 1e9, -5, lab_, rotation=90, fontsize=8)
plt.ylim(-100, 5); plt.xlabel("frequency (GHz)"); plt.ylabel("preselector response (dB)"); plt.show()

LO 2.75 GHz, image 3.05 GHz, order for 60 dB: 3
  (m, n) = (1, 3): 1.0167 GHz, preselector  101.3 dB
  (m, n) = (1, 2): 1.2250 GHz, preselector   93.9 dB
  (m, n) = (2, 4): 1.3000 GHz, preselector   91.2 dB
  (m, n) = (2, 4): 1.4500 GHz, preselector   85.8 dB
  (m, n) = (1, 2): 1.5250 GHz, preselector   82.9 dB
  (m, n) = (2, 3): 1.7333 GHz, preselector   74.3 dB
  (m, n) = (2, 3): 1.9333 GHz, preselector   64.1 dB
  (m, n) = (3, 4): 1.9875 GHz, preselector   60.8 dB
  (m, n) = (3, 4): 2.1375 GHz, preselector   49.6 dB
  (m, n) = (2, 2): 2.6000 GHz, preselector   27.9 dB
  (m, n) = (3, 3): 2.6500 GHz, preselector   35.1 dB
  (m, n) = (4, 4): 2.6750 GHz, preselector   38.1 dB
  (m, n) = (4, 4): 2.8250 GHz, preselector   50.7 dB
  (m, n) = (3, 3): 2.8500 GHz, preselector   52.3 dB
  (m, n) = (2, 2): 2.9000 GHz, preselector   55.2 dB
  (m, n) = (4, 3): 3.5667 GHz, preselector   76.5 dB
  (m, n) = (4, 3): 3.7667 GHz, preselector   80.2 dB
  (m, n) = (3, 2): 3.9750 GHz, preselector   83.5 d

## Part 2 · I/Q mismatch in a zero-IF OFDM receiver

*Predict* the EVM for 1% gain and 1° phase error.

In [3]:
print(f"simulated EVM {trx.ofdm_iq_evm_db(0.01, 1.0):.2f} dB, -IRR {-trx.irr_db(0.01, 1.0):.2f} dB")
rng = np.random.default_rng(1)
lv = np.array([-3, -1, 1, 3])
X = np.zeros(256, complex); k = np.r_[1:101, 156:256]
X[k] = rng.choice(lv, k.size) + 1j * rng.choice(lv, k.size)
y, mu, nu = trx.iq_mismatch(np.fft.ifft(X), 0.05, 5.0)       # exaggerated to make the cloud visible
Y = np.fft.fft(y) / mu
plt.figure(); plt.plot(Y[k].real, Y[k].imag, ".", ms=3); plt.gca().set_aspect("equal")
plt.title("16-QAM after 5 % and 5 degrees of I/Q mismatch"); plt.show()

simulated EVM -39.96 dB, -IRR -39.96 dB


## Part 3 · A complex filter

*Predict* the response at +1 MHz and −1 MHz of a first-order complex filter with a 0.5 MHz pole.

In [4]:
s = ngspice("complexfilter.cir")
print({k: round(s[k], 3) for k in ("pass", "rej")})
f = np.linspace(-4e6, 4e6, 801)
plt.figure()
for n in (1, 2, 3, 4):
    plt.plot(f / 1e6, [trx.complex_filter_db(x, 1e6, 0.5e6, n) for x in f], label=f"order {n}")
plt.axvline(-1, ls=":", c="k"); plt.ylim(-60, 3); plt.xlabel("frequency (MHz)"); plt.ylabel("|H| (dB)"); plt.legend(); plt.show()

{'pass': -0.0, 'rej': 12.304}


## Part 4 · Jitter

*Predict* the SNR of a 3.5 GHz tone sampled at 2949.12 MSPS with 50 fs of rms jitter.

In [5]:
print(f"aliased to {trx.nyquist_alias(3.5e9, 2949.12e6)[0]/1e6:.2f} MHz in zone {trx.nyquist_alias(3.5e9, 2949.12e6)[1]}")
print(f"simulated {trx.jitter_snr_sim(3.5e9, 2949.12e6, 50e-15):.2f} dB, limit {trx.jitter_snr_db(3.5e9, 50e-15):.2f} dB")
fin = np.geomspace(1e8, 3e10, 200)
plt.figure()
for sj in (25e-15, 50e-15, 100e-15):
    plt.semilogx(fin / 1e9, [trx.jitter_snr_db(x, sj) for x in fin], label=f"{sj*1e15:.0f} fs")
plt.xlabel("input frequency (GHz)"); plt.ylabel("jitter-limited SNR (dB)"); plt.legend(); plt.show()

aliased to 550.88 MHz in zone 3
simulated 59.13 dB, limit 59.18 dB


## Part 5 · Digital downconversion

*Predict* the in-band SNR after selecting a 20 MHz channel from a 3 GS/s stream with 60 dB full-band SNR.

In [6]:
f_alias, snr_full, snr_in = trx.ddc_snr()
print(f"alias {f_alias/1e6:.0f} MHz; full band {snr_full:.1f} dB; in band {snr_in:.2f} dB; ideal processing gain {trx.processing_gain_db(3e9, 20e6):.2f} dB")

alias 550 MHz; full band 60.0 dB; in band 79.66 dB; ideal processing gain 18.75 dB


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.